# A micrograd neuron, via SKaiNET's NN DSL

Same tribute to Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) as [`MicrogradNeuron.ipynb`](./MicrogradNeuron.ipynb), but built with SKaiNET's high-level neural-network DSL (`sequential<T, V> { input(n); dense(m); activation { … } }`) instead of the low-level DAG DSL.

Original snippet:

```python
from micrograd import nn
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]
y = n(x)
dot = draw_dot(y)
```

**How rendering works.** A `sequential { … }` builder produces a `Module<T, V>`, not a symbolic DAG — there's no graph to render until a forward pass actually executes ops. The notebook-side `Module<T, V>.asDot(input)` helper runs one forward pass under a *recording* `DefaultGraphExecutionContext` (with shape-only `VoidTensorOps`), captures every op into an `ExecutionTape`, lowers the tape to a `ComputeGraph`, and renders DOT via SKaiNET's `toGraphviz` exporter. This is the same gradient-tracer machinery that SKaiNET's autograd backprop relies on — visualizing here exercises the recording path that backward passes will reuse.

In [ ]:
USE {
    repositories {
        mavenCentral()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

## Build the neuron with the NN DSL

`nn.Neuron(2)` is one fully-connected layer from 2 inputs to 1 output, followed by an activation. micrograd defaults to `tanh`. SKaiNET 0.25.0 doesn't ship `tanh` as a `TensorOps` primitive, so the notebook integration provides a polyfill that composes the exact identity `tanh(x) = 2*sigmoid(2x) - 1` (see [`docs/upstream/tanh-activation.md`](../../docs/upstream/tanh-activation.md) for the upstream proposal that would replace it with a real primitive). The rendered graph shows the `mulScalar -> sigmoid -> mulScalar -> subScalar` decomposition rather than a single `tanh` block — faithful to what the kernel is actually computing today.

In [2]:
import sk.ainet.lang.types.FP32
import sk.ainet.lang.nn.dsl.sequential

val ctx = DefaultNeuralNetworkExecutionContext()

// n = nn.Neuron(2)
val n = sequential<FP32, Float> {
    input(2)
    dense(1)
    activation { it.tanh() }          // polyfilled as 2*sigmoid(2x)-1 (see docs/upstream/tanh-activation.md)
}

// x = [Value(1.0), Value(-2.0)]
val x = tensor<FP32, Float>(ctx, FP32::class) {
    tensor { shape(1, 2) { fromArray(floatArrayOf(1.0f, -2.0f)) } }
}

// dot = draw_dot(y)
val d = n.asDot(x)

In [6]:
d.source

digraph {
    rankdir=LR;
    n0_matmul [label="matmul | n0_matmul", shape=record];
    n0_matmul_op [label="input_shapes: [Shape: Dimensions = [1 x 2], Size (Volume) = 2, Shape: Dimensions = [2 x 1], Size (Volume) = 2]\ninput_dtypes: [FP32, FP32]\noutput_shapes: [Shape: Dimensions = [1 x 1], Size (Volume) = 1]\noutput_dtypes: [FP32]\ninputShapes: [[1, 2], [2, 1]]\noutputShapes: [[1, 1]]\ninputDTypes: [FP32, FP32]\noutputDTypes: [FP32]\ninputShape: [1, 2]\noutputShape: [1, 1]\nweights: [F@4563ee5d", shape=box, style=dashed];
    n0_matmul_op -> n0_matmul [style=dotted];
    n1_add [label="add | n1_add", shape=record];
    n1_add_op [label="input_shapes: [Shape: Dimensions = [1 x 1], Size (Volume) = 1, Shape: Dimensions = [1], Size (Volume) = 1]\ninput_dtypes: [FP32, FP32]\noutput_shapes: [Shape: Dimensions = [1 x 1], Size (Volume) = 1]\noutput_dtypes: [FP32]\ninputShapes: [[1, 1], [1]]\noutputShapes: [[1, 1]]\ninputDTypes: [FP32, FP32]\noutputDTypes: [FP32]\ninputShape: [1, 1]\noutputS